# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_08 — Probability of Backtest Overfitting

### Research question

How can we estimate the probability that a strategy-selection process is overfitting, by measuring how often the best in-sample strategy performs poorly out of sample?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
05_06_performance_haircut_and_deflated_sharpe
05_07_purged_kfold_and_embargo_cv
```

The previous notebook handled leakage from overlapping labels. This notebook addresses a different problem:

> Even without leakage, if we test many strategy variants, the best in-sample result may be an overfit winner.

This notebook covers:

1. what backtest overfitting means;
2. strategy libraries and researcher degrees of freedom;
3. combinatorially symmetric cross-validation;
4. in-sample winner selection;
5. out-of-sample rank of the in-sample winner;
6. rank-logit statistic;
7. Probability of Backtest Overfitting;
8. degradation from in-sample to out-of-sample;
9. true-alpha versus noise strategies in simulation;
10. PBO sensitivity to number of trials;
11. PBO sensitivity to performance metric;
12. PBO under all-noise placebo libraries;
13. PBO under stronger-alpha libraries;
14. selection concentration;
15. rank-consistency diagnostics;
16. performance haircut from PBO;
17. governance flags;
18. saved outputs and manifest.

The key idea:

> PBO asks: when your research process picks the best in-sample strategy, how often does that winner rank below the median out of sample?

## 1. The overfitting problem

Suppose we test $N$ strategy variants.

For each strategy $i$, we estimate an in-sample score:

$$
S_i^{IS}
$$

and select:

$$
i^* = \arg\max_i S_i^{IS}
$$

If $N$ is large, $i^*$ may be a noise winner.

The real question is not:

> What was the best in-sample score?

The real question is:

> How did the in-sample winner rank out of sample?

## 2. Combinatorially Symmetric Cross-Validation

Split the full history into $S$ equal contiguous blocks.

For each combination of $S/2$ blocks:

- use those blocks as in-sample;
- use the remaining blocks as out-of-sample.

Then:

1. compute each strategy's in-sample score;
2. choose the in-sample winner;
3. compute all strategies' out-of-sample scores;
4. rank the in-sample winner out of sample.

Repeat across many train/test combinations.

This produces a distribution of out-of-sample ranks for the in-sample winner.

## 3. Rank-logit statistic

Let $w$ be the out-of-sample percentile rank of the in-sample winner, where:

- $w=1$: best out-of-sample;
- $w=0.5$: median;
- $w=0$: worst.

Define:

$$
\lambda = \log\Big(\frac{w}{1-w}\Big)
$$

If:

$$
\lambda < 0
$$

then:

$$
w < 0.5
$$

so the in-sample winner ranked below median out of sample.

The Probability of Backtest Overfitting is:

$$
PBO = P(\lambda < 0)
$$

## 4. Interpretation

Typical reading:

| PBO | Interpretation |
|---:|---|
| near 0 | selection process is relatively stable |
| 0.20–0.40 | moderate overfitting risk |
| 0.40–0.60 | severe uncertainty |
| above 0.60 | selection process is probably selecting noise |

PBO does not prove a strategy works.

It estimates whether the **selection process** is likely to choose overfit winners.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class PBOConfig:
    n_days: int = 1_800
    annualisation: int = 252
    seed: int = 42
    n_strategies: int = 360
    n_true_alpha: int = 18
    n_partitions: int = 12
    max_cscv_combinations: int = 2_000
    cvar_alpha: float = 0.95
    transaction_cost_bps_low: float = 1.0
    transaction_cost_bps_high: float = 8.0
    pbo_warning_threshold: float = 0.40
    pbo_failure_threshold: float = 0.60
    min_oos_percentile: float = 1e-6

config = PBOConfig()
config

## 6. Simulate a strategy library

We simulate many candidate strategy variants.

Most are noise.

A small subset has weak positive true alpha.

All strategies also have:

- market beta;
- tail loading;
- turnover cost;
- volatility regimes;
- stress episodes;
- non-normal returns.

This lets us check whether PBO identifies unstable selection.

In [ ]:
def simulate_strategy_library(
    config: PBOConfig,
    scenario_name: str = "mixed_true_alpha",
    true_alpha_scale: float = 1.0,
    all_noise: bool = False,
):
    rng = np.random.default_rng(config.seed + abs(hash(scenario_name)) % 100_000)
    dates = pd.bdate_range("2016-01-01", periods=config.n_days)

    strategy_ids = [f"strategy_{i:03d}" for i in range(config.n_strategies)]
    n = config.n_strategies

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    market = rng.standard_t(df=6, size=config.n_days) * 0.010
    style = rng.standard_t(df=6, size=config.n_days) * 0.006
    carry = rng.standard_t(df=6, size=config.n_days) * 0.005

    for t in range(1, config.n_days):
        if regime[t - 1] == 0:
            regime[t] = rng.choice([0, 1], p=[0.985, 0.015])
        else:
            regime[t] = rng.choice([0, 1], p=[0.060, 0.940])

        if regime[t] == 1:
            market[t] *= 2.5
            style[t] *= 2.0
            carry[t] *= 1.8

        u = rng.uniform()
        if u < 0.010:
            stress_type[t] = "left_tail_event"
            market[t] -= rng.uniform(0.020, 0.080)
        elif u < 0.016:
            stress_type[t] = "right_tail_event"
            market[t] += rng.uniform(0.015, 0.050)

    true_alpha_daily = np.zeros(n)
    alpha_indices = np.array([], dtype=int)

    if not all_noise and config.n_true_alpha > 0:
        alpha_indices = rng.choice(n, size=config.n_true_alpha, replace=False)
        true_alpha_daily[alpha_indices] = rng.uniform(0.00006, 0.00022, size=config.n_true_alpha) * true_alpha_scale

    beta_market = rng.normal(0.10, 0.35, size=n)
    beta_style = rng.normal(0.05, 0.25, size=n)
    beta_carry = rng.normal(0.02, 0.20, size=n)

    idio_vol = rng.uniform(0.005, 0.018, size=n)
    tail_loading = rng.uniform(0.0, 0.018, size=n)
    turnover = rng.uniform(0.05, 0.90, size=n)
    cost_bps = rng.uniform(config.transaction_cost_bps_low, config.transaction_cost_bps_high, size=n)

    returns = np.zeros((config.n_days, n))

    for t in range(config.n_days):
        vol_mult = 1.0 if regime[t] == 0 else 2.2
        common = (
            beta_market * market[t]
            + beta_style * style[t]
            + beta_carry * carry[t]
        )

        idio = rng.standard_t(df=5, size=n) * np.sqrt((5 - 2) / 5) * idio_vol * vol_mult

        tail = np.zeros(n)
        if stress_type[t] == "left_tail_event":
            tail = -tail_loading * rng.uniform(0.5, 1.2, size=n)
        elif stress_type[t] == "right_tail_event":
            tail = 0.35 * tail_loading * rng.uniform(0.2, 0.8, size=n)

        cost_drag = turnover * cost_bps / 10000.0 / 10.0

        returns[t] = true_alpha_daily + common + idio + tail - cost_drag

    returns_df = pd.DataFrame(returns, index=dates, columns=strategy_ids)

    metadata = pd.DataFrame({
        "strategy_id": strategy_ids,
        "true_alpha_daily": true_alpha_daily,
        "true_alpha_ann": true_alpha_daily * config.annualisation,
        "has_true_alpha": true_alpha_daily > 0,
        "beta_market": beta_market,
        "beta_style": beta_style,
        "beta_carry": beta_carry,
        "idio_vol_daily": idio_vol,
        "tail_loading": tail_loading,
        "turnover": turnover,
        "cost_bps": cost_bps,
        "scenario_name": scenario_name,
    })

    regime_info = pd.DataFrame({
        "market": market,
        "style": style,
        "carry": carry,
        "regime": regime,
        "stress_type": stress_type,
    }, index=dates)

    return returns_df, metadata, regime_info

strategy_returns, strategy_metadata, regime_info = simulate_strategy_library(config, "mixed_true_alpha")

strategy_returns.head(), strategy_metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in strategy_returns.columns[:12]:
    plt.plot(strategy_returns.index, (1 + strategy_returns[col]).cumprod(), alpha=0.65, label=col)
plt.title("Example Strategy Equity Curves")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=3)
plt.show()

## 7. Performance metrics

PBO can be computed using any selection metric.

We implement:

- Sharpe;
- Sortino;
- Calmar;
- annualised return.

The selection metric matters. A process that selects based on a fragile metric may overfit more.

In [ ]:
def max_drawdown_from_returns(returns):
    r = pd.Series(returns).dropna()
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return float(dd.min()) if len(dd) else np.nan

def annualised_return(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return np.nan
    return float((1 + r).prod() ** (annualisation / len(r)) - 1)

def annualised_vol(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std(ddof=1) * np.sqrt(annualisation))

def sharpe_ratio(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    vol = r.std(ddof=1)
    if len(r) < 2 or vol == 0:
        return np.nan
    return float(r.mean() / vol * np.sqrt(annualisation))

def sortino_ratio(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    downside = r[r < 0]
    if len(downside) < 2 or downside.std(ddof=1) == 0:
        return np.nan
    return float(r.mean() / downside.std(ddof=1) * np.sqrt(annualisation))

def calmar_ratio(returns, annualisation=252):
    ann = annualised_return(returns, annualisation)
    mdd = max_drawdown_from_returns(returns)
    if not np.isfinite(mdd) or mdd >= 0:
        return np.nan
    return float(ann / abs(mdd))

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def compute_metric(returns, metric_name, annualisation=252):
    if metric_name == "sharpe":
        return sharpe_ratio(returns, annualisation)
    if metric_name == "sortino":
        return sortino_ratio(returns, annualisation)
    if metric_name == "calmar":
        return calmar_ratio(returns, annualisation)
    if metric_name == "annualised_return":
        return annualised_return(returns, annualisation)
    raise ValueError(f"Unknown metric: {metric_name}")

def strategy_performance_table(returns_df, config):
    rows = []

    for col in returns_df.columns:
        r = returns_df[col].dropna()
        var, cvar = historical_var_cvar(-r, config.cvar_alpha)
        rows.append({
            "strategy_id": col,
            "n_obs": len(r),
            "annualised_return": annualised_return(r, config.annualisation),
            "annualised_vol": annualised_vol(r, config.annualisation),
            "sharpe": sharpe_ratio(r, config.annualisation),
            "sortino": sortino_ratio(r, config.annualisation),
            "calmar": calmar_ratio(r, config.annualisation),
            "max_drawdown": max_drawdown_from_returns(r),
            "skew": float(r.skew()),
            "excess_kurtosis": float(r.kurtosis()),
            "VaR": var,
            "CVaR": cvar,
            "total_return": float((1 + r).prod() - 1),
        })

    return pd.DataFrame(rows)

full_perf = strategy_performance_table(strategy_returns, config).merge(strategy_metadata, on="strategy_id", how="left")
full_perf.sort_values("sharpe", ascending=False).head(15)

## 8. Create CSCV partitions

Split the date index into contiguous partitions.

For $S=12$, CSCV uses combinations of 6 in-sample partitions and 6 out-of-sample partitions.

There are:

$$
\binom{12}{6}=924
$$

combinations, which is manageable.

In [ ]:
def make_contiguous_partitions(index, n_partitions):
    n = len(index)
    positions = np.arange(n)
    groups = np.array_split(positions, n_partitions)

    rows = []
    for group_id, pos in enumerate(groups):
        rows.append({
            "partition_id": group_id,
            "start_idx": int(pos[0]),
            "end_idx": int(pos[-1]),
            "n_obs": len(pos),
            "start_date": index[pos[0]],
            "end_date": index[pos[-1]],
        })

    return groups, pd.DataFrame(rows)

partitions, partition_table = make_contiguous_partitions(strategy_returns.index, config.n_partitions)

partition_table

## 9. CSCV combinations

Each CSCV split chooses half the partitions as in-sample.

The complement is out-of-sample.

If the number of combinations is too large, the implementation samples combinations reproducibly.

In [ ]:
def make_cscv_combinations(n_partitions, max_combinations, seed):
    half = n_partitions // 2
    all_combos = list(combinations(range(n_partitions), half))

    if len(all_combos) <= max_combinations:
        selected = all_combos
    else:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(all_combos), size=max_combinations, replace=False)
        selected = [all_combos[i] for i in sorted(idx)]

    rows = []
    for split_id, in_sample_parts in enumerate(selected):
        in_sample_parts = tuple(in_sample_parts)
        out_sample_parts = tuple(p for p in range(n_partitions) if p not in in_sample_parts)
        rows.append({
            "split_id": split_id,
            "in_sample_partitions": in_sample_parts,
            "out_sample_partitions": out_sample_parts,
        })

    return pd.DataFrame(rows)

cscv_combos = make_cscv_combinations(config.n_partitions, config.max_cscv_combinations, config.seed)

pd.Series({
    "n_cscv_combinations": len(cscv_combos),
    "first_combo": str(cscv_combos.iloc[0].to_dict()),
})

## 10. Precompute partition-level returns

To make CSCV efficient, we precompute metric inputs by partition.

For each split, we concatenate selected partitions.

In [ ]:
def indices_for_partitions(partitions, partition_ids):
    arrays = [partitions[p] for p in partition_ids]
    return np.concatenate(arrays)

def returns_for_partitions(returns_df, partitions, partition_ids):
    idx = indices_for_partitions(partitions, partition_ids)
    return returns_df.iloc[idx]

def metric_by_strategy(returns_df, metric_name, config):
    values = {}
    for col in returns_df.columns:
        values[col] = compute_metric(returns_df[col], metric_name, config.annualisation)
    return pd.Series(values)

# Example.
example_is = returns_for_partitions(strategy_returns, partitions, cscv_combos.iloc[0]["in_sample_partitions"])
metric_by_strategy(example_is, "sharpe", config).head()

## 11. CSCV / PBO engine

For each CSCV split:

1. score every strategy in sample;
2. select best in-sample strategy;
3. score every strategy out of sample;
4. rank selected strategy out of sample;
5. convert out-of-sample rank to logit;
6. record degradation.

Goodness percentile:

$$
w = 1 - \frac{rank-1}{N-1}
$$

where best rank is 1.

Then:

$$
\lambda = \log\frac{w}{1-w}
$$

In [ ]:
def goodness_percentile_from_rank(rank, n_models, min_eps=1e-6):
    if n_models <= 1:
        return 0.5
    w = 1.0 - (rank - 1.0) / (n_models - 1.0)
    return float(np.clip(w, min_eps, 1.0 - min_eps))

def logit(x):
    x = np.clip(x, 1e-12, 1 - 1e-12)
    return float(np.log(x / (1 - x)))

def run_cscv_pbo(returns_df, partitions, cscv_combos, metric_name, config):
    rows = []

    n_models = returns_df.shape[1]
    strategy_ids = list(returns_df.columns)

    for _, combo in cscv_combos.iterrows():
        split_id = int(combo["split_id"])
        is_parts = combo["in_sample_partitions"]
        oos_parts = combo["out_sample_partitions"]

        is_returns = returns_for_partitions(returns_df, partitions, is_parts)
        oos_returns = returns_for_partitions(returns_df, partitions, oos_parts)

        is_scores = metric_by_strategy(is_returns, metric_name, config).replace([np.inf, -np.inf], np.nan)
        oos_scores = metric_by_strategy(oos_returns, metric_name, config).replace([np.inf, -np.inf], np.nan)

        if is_scores.dropna().empty or oos_scores.dropna().empty:
            continue

        selected_strategy = is_scores.idxmax()
        selected_is_score = is_scores.loc[selected_strategy]
        selected_oos_score = oos_scores.loc[selected_strategy]

        # Rank descending: 1 is best.
        oos_ranks = oos_scores.rank(ascending=False, method="min")
        selected_oos_rank = float(oos_ranks.loc[selected_strategy])
        oos_goodness = goodness_percentile_from_rank(selected_oos_rank, n_models, config.min_oos_percentile)
        lambda_logit = logit(oos_goodness)

        is_rank = float(is_scores.rank(ascending=False, method="min").loc[selected_strategy])

        rows.append({
            "split_id": split_id,
            "metric": metric_name,
            "selected_strategy": selected_strategy,
            "selected_is_score": selected_is_score,
            "selected_oos_score": selected_oos_score,
            "is_rank": is_rank,
            "oos_rank": selected_oos_rank,
            "oos_goodness_percentile": oos_goodness,
            "lambda_logit": lambda_logit,
            "is_minus_oos_score": selected_is_score - selected_oos_score,
            "overfit_indicator": lambda_logit < 0,
            "in_sample_partitions": str(is_parts),
            "out_sample_partitions": str(oos_parts),
        })

    pbo_results = pd.DataFrame(rows)

    summary = pd.DataFrame([{
        "metric": metric_name,
        "n_splits": len(pbo_results),
        "pbo": float(pbo_results["overfit_indicator"].mean()),
        "mean_lambda": float(pbo_results["lambda_logit"].mean()),
        "median_lambda": float(pbo_results["lambda_logit"].median()),
        "mean_oos_goodness_percentile": float(pbo_results["oos_goodness_percentile"].mean()),
        "median_oos_goodness_percentile": float(pbo_results["oos_goodness_percentile"].median()),
        "mean_is_score": float(pbo_results["selected_is_score"].mean()),
        "mean_oos_score": float(pbo_results["selected_oos_score"].mean()),
        "mean_score_degradation": float(pbo_results["is_minus_oos_score"].mean()),
    }])

    return pbo_results, summary

pbo_sharpe_results, pbo_sharpe_summary = run_cscv_pbo(
    strategy_returns,
    partitions,
    cscv_combos,
    "sharpe",
    config,
)

pbo_sharpe_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(pbo_sharpe_results["lambda_logit"], bins=50)
plt.axvline(0, linestyle="--", label="median OOS rank boundary")
plt.title("CSCV Rank-Logit Distribution, Sharpe Selection")
plt.xlabel("Logit OOS rank of IS winner")
plt.ylabel("Count")
plt.legend()
plt.show()

## 12. In-sample versus out-of-sample score degradation

A large average gap between in-sample and out-of-sample performance indicates overfit selection.

In [ ]:
degradation_summary = pbo_sharpe_results[[
    "selected_is_score",
    "selected_oos_score",
    "is_minus_oos_score",
    "oos_goodness_percentile",
    "lambda_logit",
]].describe().T

degradation_summary

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(pbo_sharpe_results["selected_is_score"], pbo_sharpe_results["selected_oos_score"], alpha=0.35)
plt.axline((0, 0), slope=1, linestyle="--")
plt.title("Selected Strategy IS Score vs OOS Score")
plt.xlabel("In-sample selected Sharpe")
plt.ylabel("Out-of-sample Sharpe")
plt.show()

## 13. Which strategies get selected?

If one robust strategy is repeatedly selected and performs well OOS, that is healthier than random rotation among many noisy winners.

In [ ]:
selection_frequency = (
    pbo_sharpe_results
    .groupby("selected_strategy")
    .agg(
        n_selected=("split_id", "count"),
        mean_is_score=("selected_is_score", "mean"),
        mean_oos_score=("selected_oos_score", "mean"),
        mean_oos_goodness=("oos_goodness_percentile", "mean"),
        pbo_rate=("overfit_indicator", "mean"),
    )
    .reset_index()
    .merge(strategy_metadata[["strategy_id", "has_true_alpha", "true_alpha_ann", "turnover", "tail_loading"]], left_on="selected_strategy", right_on="strategy_id", how="left")
    .sort_values("n_selected", ascending=False)
)

selection_frequency.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = selection_frequency.head(20).sort_values("n_selected")
plt.barh(plot_df["selected_strategy"], plot_df["n_selected"])
plt.title("Most Frequently Selected In-Sample Winners")
plt.xlabel("Selection count")
plt.ylabel("Strategy")
plt.show()

## 14. Selection accuracy in simulation

Because this is synthetic, we know which strategies truly have alpha.

In real research, we do not know this.

This diagnostic shows whether CSCV selection tends to choose real-alpha strategies or noise winners.

In [ ]:
selection_truth_report = pd.DataFrame([{
    "n_unique_selected": selection_frequency["selected_strategy"].nunique(),
    "selected_true_alpha_rate_weighted_by_frequency": np.average(
        selection_frequency["has_true_alpha"].astype(float),
        weights=selection_frequency["n_selected"]
    ),
    "selected_true_alpha_rate_unique": selection_frequency["has_true_alpha"].mean(),
    "universe_true_alpha_rate": strategy_metadata["has_true_alpha"].mean(),
    "top_full_sample_sharpe_has_true_alpha": bool(full_perf.sort_values("sharpe", ascending=False).iloc[0]["has_true_alpha"]),
}])

selection_truth_report

## 15. PBO by performance metric

Different objective functions can have different overfitting risk.

We compare:

- Sharpe;
- Sortino;
- Calmar;
- annualised return.

In [ ]:
metric_results = []
metric_summaries = []

for metric_name in ["sharpe", "sortino", "calmar", "annualised_return"]:
    res, summ = run_cscv_pbo(strategy_returns, partitions, cscv_combos, metric_name, config)
    metric_results.append(res)
    metric_summaries.append(summ)

pbo_metric_results = pd.concat(metric_results, ignore_index=True)
pbo_metric_summary = pd.concat(metric_summaries, ignore_index=True).sort_values("pbo")

pbo_metric_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(pbo_metric_summary["metric"], pbo_metric_summary["pbo"])
plt.axvline(config.pbo_warning_threshold, linestyle="--", label="warning")
plt.axvline(config.pbo_failure_threshold, linestyle=":", label="failure")
plt.title("PBO by Selection Metric")
plt.xlabel("Probability of Backtest Overfitting")
plt.ylabel("Metric")
plt.legend()
plt.show()

## 16. PBO sensitivity to number of trials

As more strategy variants are tested, the chance of selecting a noise winner rises.

We estimate PBO using random subsets of the strategy library.

In [ ]:
def pbo_for_strategy_subset(returns_df, subset_size, metric_name, config, seed_offset=0):
    rng = np.random.default_rng(config.seed + 10_000 + seed_offset)
    cols = rng.choice(returns_df.columns, size=min(subset_size, returns_df.shape[1]), replace=False)
    subset = returns_df[list(cols)]
    res, summ = run_cscv_pbo(subset, partitions, cscv_combos, metric_name, config)
    out = summ.iloc[0].to_dict()
    out["subset_size"] = subset_size
    return out

subset_sizes = [20, 50, 100, 180, 260, 360]
trial_sensitivity_rows = []

for subset_size in subset_sizes:
    # Repeat a few random subsets for stability.
    for rep in range(5):
        trial_sensitivity_rows.append(pbo_for_strategy_subset(strategy_returns, subset_size, "sharpe", config, seed_offset=100 * subset_size + rep))

pbo_trial_sensitivity = pd.DataFrame(trial_sensitivity_rows)

pbo_trial_summary = (
    pbo_trial_sensitivity
    .groupby("subset_size")
    .agg(
        mean_pbo=("pbo", "mean"),
        std_pbo=("pbo", "std"),
        mean_degradation=("mean_score_degradation", "mean"),
        mean_oos_percentile=("mean_oos_goodness_percentile", "mean"),
    )
    .reset_index()
)

pbo_trial_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(
    pbo_trial_summary["subset_size"],
    pbo_trial_summary["mean_pbo"],
    yerr=pbo_trial_summary["std_pbo"],
    marker="o",
    capsize=4,
)
plt.axhline(config.pbo_warning_threshold, linestyle="--", label="warning")
plt.axhline(config.pbo_failure_threshold, linestyle=":", label="failure")
plt.title("PBO Sensitivity to Number of Tried Strategies")
plt.xlabel("Number of strategy trials")
plt.ylabel("PBO")
plt.legend()
plt.show()

## 17. Placebo all-noise library

Now simulate a library where no strategy has true alpha.

PBO should generally be high because in-sample winners are noise.

In [ ]:
noise_returns, noise_metadata, noise_regime = simulate_strategy_library(
    config,
    scenario_name="all_noise_placebo",
    all_noise=True,
)

noise_pbo_results, noise_pbo_summary = run_cscv_pbo(
    noise_returns,
    partitions,
    cscv_combos,
    "sharpe",
    config,
)

noise_pbo_summary

## 18. Stronger-alpha library

Now simulate a library with stronger true alpha.

PBO should improve if genuine signals dominate noise.

In [ ]:
strong_returns, strong_metadata, strong_regime = simulate_strategy_library(
    config,
    scenario_name="stronger_true_alpha",
    true_alpha_scale=2.5,
    all_noise=False,
)

strong_pbo_results, strong_pbo_summary = run_cscv_pbo(
    strong_returns,
    partitions,
    cscv_combos,
    "sharpe",
    config,
)

scenario_pbo_summary = pd.concat([
    pbo_sharpe_summary.assign(scenario="mixed_true_alpha"),
    noise_pbo_summary.assign(scenario="all_noise_placebo"),
    strong_pbo_summary.assign(scenario="stronger_true_alpha"),
], ignore_index=True)

scenario_pbo_summary[["scenario", "pbo", "mean_oos_goodness_percentile", "mean_score_degradation", "mean_is_score", "mean_oos_score"]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(scenario_pbo_summary["scenario"], scenario_pbo_summary["pbo"])
plt.axvline(config.pbo_warning_threshold, linestyle="--", label="warning")
plt.axvline(config.pbo_failure_threshold, linestyle=":", label="failure")
plt.title("PBO by Simulation Scenario")
plt.xlabel("PBO")
plt.ylabel("Scenario")
plt.legend()
plt.show()

## 19. Rank consistency diagnostics

A robust strategy selection process should have some consistency between in-sample and out-of-sample ranks.

We measure rank correlation across CSCV splits.

In [ ]:
def rank_correlation_diagnostics(returns_df, partitions, cscv_combos, metric_name, config):
    rows = []

    for _, combo in cscv_combos.iterrows():
        is_returns = returns_for_partitions(returns_df, partitions, combo["in_sample_partitions"])
        oos_returns = returns_for_partitions(returns_df, partitions, combo["out_sample_partitions"])

        is_scores = metric_by_strategy(is_returns, metric_name, config)
        oos_scores = metric_by_strategy(oos_returns, metric_name, config)

        joined = pd.concat([is_scores.rename("is_score"), oos_scores.rename("oos_score")], axis=1).dropna()
        if len(joined) < 3:
            corr = np.nan
            rank_corr = np.nan
        else:
            corr = joined["is_score"].corr(joined["oos_score"])
            rank_corr = joined["is_score"].rank().corr(joined["oos_score"].rank())

        rows.append({
            "split_id": int(combo["split_id"]),
            "metric": metric_name,
            "score_correlation": corr,
            "rank_correlation": rank_corr,
        })

    return pd.DataFrame(rows)

rank_consistency = rank_correlation_diagnostics(strategy_returns, partitions, cscv_combos, "sharpe", config)

rank_consistency.describe().T

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(rank_consistency["rank_correlation"].dropna(), bins=40)
plt.axvline(0, linestyle="--")
plt.title("IS/OOS Rank Correlation Across CSCV Splits")
plt.xlabel("Rank correlation")
plt.ylabel("Count")
plt.show()

## 20. Performance haircut from PBO

A practical reporting haircut can use PBO:

$$
\begin{aligned}
SR_{PBO-adjusted} &= SR_{observed} \\
&\quad - PBO \times penalty
\end{aligned}
$$

This is not a universal formula.

It is a transparent governance penalty for selection risk.

In [ ]:
observed_best = full_perf.sort_values("sharpe", ascending=False).iloc[0]
observed_best_strategy = observed_best["strategy_id"]

pbo = pbo_sharpe_summary["pbo"].iloc[0]
mean_degradation = pbo_sharpe_summary["mean_score_degradation"].iloc[0]

pbo_penalty = pbo * max(0.0, mean_degradation)
conservative_sharpe = observed_best["sharpe"] - pbo_penalty

pbo_haircut_report = pd.DataFrame([{
    "selected_strategy_full_sample_best": observed_best_strategy,
    "observed_full_sample_sharpe": observed_best["sharpe"],
    "pbo": pbo,
    "mean_is_oos_degradation": mean_degradation,
    "pbo_penalty": pbo_penalty,
    "pbo_adjusted_sharpe": conservative_sharpe,
    "has_true_alpha_in_simulation": bool(observed_best["has_true_alpha"]),
}])

pbo_haircut_report

## 21. Governance flags

A research process should be reviewed if:

1. PBO exceeds warning or failure thresholds;
2. in-sample winners rank below median OOS too often;
3. in-sample to OOS degradation is large;
4. selected winners rotate randomly;
5. rank consistency is weak or negative;
6. all-noise placebo has similar performance to real library;
7. PBO-adjusted Sharpe is weak.

In [ ]:
top_selection_share = selection_frequency["n_selected"].iloc[0] / len(pbo_sharpe_results)
rank_corr_mean = rank_consistency["rank_correlation"].mean()
noise_pbo = noise_pbo_summary["pbo"].iloc[0]
strong_pbo = strong_pbo_summary["pbo"].iloc[0]

governance_flags = pd.DataFrame([{
    "metric": "sharpe",
    "pbo": pbo,
    "mean_oos_goodness_percentile": pbo_sharpe_summary["mean_oos_goodness_percentile"].iloc[0],
    "mean_score_degradation": mean_degradation,
    "top_selection_share": top_selection_share,
    "n_unique_selected": selection_frequency["selected_strategy"].nunique(),
    "mean_rank_correlation": rank_corr_mean,
    "noise_placebo_pbo": noise_pbo,
    "strong_alpha_pbo": strong_pbo,
    "observed_full_sample_sharpe": observed_best["sharpe"],
    "pbo_adjusted_sharpe": conservative_sharpe,
    "flag_pbo_above_warning": bool(pbo > config.pbo_warning_threshold),
    "flag_pbo_above_failure": bool(pbo > config.pbo_failure_threshold),
    "flag_oos_percentile_below_median": bool(pbo_sharpe_summary["mean_oos_goodness_percentile"].iloc[0] < 0.50),
    "flag_degradation_large": bool(mean_degradation > 1.0),
    "flag_selection_unstable": bool(top_selection_share < 0.15),
    "flag_rank_correlation_negative": bool(rank_corr_mean < 0.0),
    "flag_pbo_adjusted_sharpe_below_0_5": bool(conservative_sharpe < 0.50),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_pbo_above_warning",
        "flag_pbo_above_failure",
        "flag_oos_percentile_below_median",
        "flag_degradation_large",
        "flag_selection_unstable",
        "flag_rank_correlation_negative",
        "flag_pbo_adjusted_sharpe_below_0_5",
    ]
].any(axis=1)

governance_flags

## 22. Audit checks

Numerical and process checks:

1. partitions cover the full dataset;
2. CSCV combinations use half the partitions in sample;
3. every CSCV split has complementary IS/OOS partitions;
4. OOS percentiles are in $(0,1)$;
5. PBO is in $[0,1]$;
6. lambda values are finite;
7. selected strategies exist in strategy universe;
8. placebo and strong-alpha scenarios completed.

In [ ]:
audit_rows = []

covered = np.concatenate(partitions)
partition_cover_ok = bool(np.array_equal(np.sort(covered), np.arange(len(strategy_returns))))
audit_rows.append({
    "check": "partitions_cover_full_dataset",
    "value": partition_cover_ok,
    "passed": partition_cover_ok,
})

half = config.n_partitions // 2
combo_half_ok = bool(all(len(x) == half for x in cscv_combos["in_sample_partitions"]))
audit_rows.append({
    "check": "cscv_combos_use_half_partitions",
    "value": combo_half_ok,
    "passed": combo_half_ok,
})

complements_ok = True
all_parts = set(range(config.n_partitions))
for _, row in cscv_combos.iterrows():
    complements_ok = complements_ok and (set(row["in_sample_partitions"]).union(set(row["out_sample_partitions"])) == all_parts)
    complements_ok = complements_ok and (set(row["in_sample_partitions"]).intersection(set(row["out_sample_partitions"])) == set())

audit_rows.append({
    "check": "cscv_in_sample_out_sample_are_complements",
    "value": bool(complements_ok),
    "passed": bool(complements_ok),
})

percentiles_valid = bool(((pbo_sharpe_results["oos_goodness_percentile"] > 0) & (pbo_sharpe_results["oos_goodness_percentile"] < 1)).all())
audit_rows.append({
    "check": "oos_percentiles_in_open_unit_interval",
    "value": percentiles_valid,
    "passed": percentiles_valid,
})

pbo_valid = bool(0 <= pbo <= 1)
audit_rows.append({
    "check": "pbo_in_unit_interval",
    "value": float(pbo),
    "passed": pbo_valid,
})

lambda_finite = bool(np.isfinite(pbo_sharpe_results["lambda_logit"]).all())
audit_rows.append({
    "check": "lambda_values_finite",
    "value": lambda_finite,
    "passed": lambda_finite,
})

selected_exist = bool(set(pbo_sharpe_results["selected_strategy"]).issubset(set(strategy_returns.columns)))
audit_rows.append({
    "check": "selected_strategies_exist",
    "value": selected_exist,
    "passed": selected_exist,
})

scenario_complete = bool(len(noise_pbo_results) > 0 and len(strong_pbo_results) > 0)
audit_rows.append({
    "check": "placebo_and_strong_scenarios_completed",
    "value": scenario_complete,
    "passed": scenario_complete,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 23. Practical checklist for PBO

Before trusting a research selection process:

1. **Count strategy trials**  
   More trials create more selection bias.

2. **Use CSCV or equivalent**  
   Measure how IS winners rank OOS.

3. **Look at lambda distribution**  
   PBO is the left side of the lambda distribution.

4. **Measure degradation**  
   In-sample score minus OOS score is a core diagnostic.

5. **Check selection concentration**  
   Stable repeated winners are better than random rotating winners.

6. **Compare metrics**  
   Some objectives overfit more than others.

7. **Run placebo libraries**  
   If all-noise looks similar, your process is weak.

8. **Run sensitivity to trial count**  
   PBO should not explode with small grid changes.

9. **Report PBO-adjusted performance**  
   Do not report only full-sample best Sharpe.

10. **Save every trial**  
   PBO is meaningless if failed experiments are hidden.

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/probability_of_backtest_overfitting_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "strategy_returns": output_dir / "strategy_returns.csv",
    "strategy_metadata": output_dir / "strategy_metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "full_performance": output_dir / "full_performance.csv",
    "partition_table": output_dir / "partition_table.csv",
    "cscv_combinations": output_dir / "cscv_combinations.csv",
    "pbo_sharpe_results": output_dir / "pbo_sharpe_results.csv",
    "pbo_sharpe_summary": output_dir / "pbo_sharpe_summary.csv",
    "degradation_summary": output_dir / "degradation_summary.csv",
    "selection_frequency": output_dir / "selection_frequency.csv",
    "selection_truth_report": output_dir / "selection_truth_report.csv",
    "pbo_metric_results": output_dir / "pbo_metric_results.csv",
    "pbo_metric_summary": output_dir / "pbo_metric_summary.csv",
    "pbo_trial_sensitivity": output_dir / "pbo_trial_sensitivity.csv",
    "pbo_trial_summary": output_dir / "pbo_trial_summary.csv",
    "noise_pbo_results": output_dir / "noise_pbo_results.csv",
    "noise_pbo_summary": output_dir / "noise_pbo_summary.csv",
    "strong_pbo_results": output_dir / "strong_pbo_results.csv",
    "strong_pbo_summary": output_dir / "strong_pbo_summary.csv",
    "scenario_pbo_summary": output_dir / "scenario_pbo_summary.csv",
    "rank_consistency": output_dir / "rank_consistency.csv",
    "pbo_haircut_report": output_dir / "pbo_haircut_report.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

strategy_returns.to_csv(paths["strategy_returns"])
strategy_metadata.to_csv(paths["strategy_metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
full_perf.to_csv(paths["full_performance"], index=False)
partition_table.to_csv(paths["partition_table"], index=False)

# Convert tuple columns to strings for CSV.
cscv_to_save = cscv_combos.copy()
cscv_to_save["in_sample_partitions"] = cscv_to_save["in_sample_partitions"].astype(str)
cscv_to_save["out_sample_partitions"] = cscv_to_save["out_sample_partitions"].astype(str)
cscv_to_save.to_csv(paths["cscv_combinations"], index=False)

pbo_sharpe_results.to_csv(paths["pbo_sharpe_results"], index=False)
pbo_sharpe_summary.to_csv(paths["pbo_sharpe_summary"], index=False)
degradation_summary.to_csv(paths["degradation_summary"])
selection_frequency.to_csv(paths["selection_frequency"], index=False)
selection_truth_report.to_csv(paths["selection_truth_report"], index=False)
pbo_metric_results.to_csv(paths["pbo_metric_results"], index=False)
pbo_metric_summary.to_csv(paths["pbo_metric_summary"], index=False)
pbo_trial_sensitivity.to_csv(paths["pbo_trial_sensitivity"], index=False)
pbo_trial_summary.to_csv(paths["pbo_trial_summary"], index=False)
noise_pbo_results.to_csv(paths["noise_pbo_results"], index=False)
noise_pbo_summary.to_csv(paths["noise_pbo_summary"], index=False)
strong_pbo_results.to_csv(paths["strong_pbo_results"], index=False)
strong_pbo_summary.to_csv(paths["strong_pbo_summary"], index=False)
scenario_pbo_summary.to_csv(paths["scenario_pbo_summary"], index=False)
rank_consistency.to_csv(paths["rank_consistency"], index=False)
pbo_haircut_report.to_csv(paths["pbo_haircut_report"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "probability_of_backtest_overfitting_outputs",
    "schema_version": "probability_of_backtest_overfitting_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_cscv_combinations": len(cscv_combos),
    "pbo_sharpe_summary": pbo_sharpe_summary.to_dict(orient="records"),
    "pbo_metric_summary": pbo_metric_summary.to_dict(orient="records"),
    "scenario_pbo_summary": scenario_pbo_summary.to_dict(orient="records"),
    "pbo_haircut_report": pbo_haircut_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "CSCV splits the history into contiguous partitions and uses half for IS and half for OOS.",
        "For each split, the best IS strategy is ranked OOS among all strategies.",
        "PBO is the fraction of rank-logit lambda values below zero.",
        "All-noise and stronger-alpha scenarios are included as controls.",
        "PBO sensitivity to number of tried strategies is reported.",
        "A simple PBO-based performance haircut is included for governance reporting.",
        "This notebook diagnoses selection-process overfitting, not individual strategy truth."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["pbo_sharpe_summary"], paths["pbo_metric_summary"], paths["scenario_pbo_summary"], paths["governance_flags"], paths["manifest"]

## 25. Limitations

### Synthetic data

The notebook uses simulated strategies with known true-alpha labels. Real research does not know which strategies truly have alpha.

### Independent-trial assumption is imperfect

Strategy variants are often correlated. Counting every variant as independent can overstate multiple-testing severity.

### CSCV computational cost

The number of combinations grows quickly with partitions. Large studies may need sampling.

### Performance metric dependence

PBO depends on the metric used for selection. Sharpe, Sortino, Calmar, and return can produce different PBO estimates.

### No transaction-cost uncertainty

Costs are embedded in simulated returns, but cost-estimation uncertainty is not modelled.

### No live validation

PBO is still based on historical data. Live shadow-mode monitoring is needed before production deployment.

## 26. Research frontier and extensions

Important extensions include:

1. combinatorially symmetric cross-validation with purged labels;
2. PBO under correlated strategy clusters;
3. PBO using Deflated Sharpe rather than Sharpe;
4. PBO for machine-learning hyperparameter searches;
5. PBO with transaction-cost uncertainty;
6. PBO with regime-conditioned strategy libraries;
7. experiment registry integration;
8. live PBO-style degradation monitoring;
9. PBO for portfolio allocation methods;
10. Chinese futures strategy-library PBO with night-session and contract-roll effects.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_09_strategy_capacity_and_market_impact**  
   Add capacity overfitting and impact-adjusted selection.

2. **05_10_research_audit_trail_and_manifest**  
   Record every trial needed to estimate PBO honestly.

3. **05_11_live_shadow_mode_monitoring**  
   Track whether selected strategies degrade live.

4. **05_12_full_research_to_production_checklist**  
   Combine leakage, PBO, DSR, costs, and governance.

5. **07_06_research_governance_capstone**  
   End-to-end research overfitting controls.

## 28. Summary

This notebook implemented Probability of Backtest Overfitting.

It showed how to:

1. simulate a large strategy library;
2. distinguish true-alpha and noise strategies in simulation;
3. split history into CSCV partitions;
4. generate combinatorial IS/OOS splits;
5. select in-sample winners;
6. rank winners out of sample;
7. compute rank-logit statistics;
8. estimate PBO;
9. measure IS/OOS degradation;
10. analyse selection frequency;
11. compare PBO across metrics;
12. test PBO sensitivity to number of trials;
13. run all-noise and stronger-alpha controls;
14. measure rank consistency;
15. create a PBO performance haircut;
16. generate governance flags and audit checks;
17. save all outputs and manifests.

The key computational takeaway:

> PBO is estimated from the distribution of out-of-sample ranks of in-sample winners.

The key financial takeaway:

> A research process that consistently selects strategies ranking below median out of sample is not discovering alpha; it is discovering noise.

## 29. Further reading

- Bailey et al. on Probability of Backtest Overfitting.
- López de Prado, *Advances in Financial Machine Learning.*
- Bailey and López de Prado on Deflated Sharpe Ratio.
- White's Reality Check.
- Hansen's Superior Predictive Ability test.
- Harvey, Liu and Zhu on multiple testing in factor research.
- Institutional model-risk literature on experiment registries and research governance.